In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

FILE_PATH = "time_series_data.csv"
DATE_COLUMN = "date"
TARGET = "target"

df = pd.read_csv(
    FILE_PATH,
    parse_dates=[DATE_COLUMN]
)

df = (
    df
    .sort_values(DATE_COLUMN)
    .reset_index(drop=True)
)

print(df.head())
print(df.shape)
df.info()

In [ ]:
# ============================================================
# 2. EDA
# ============================================================

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

plt.figure(figsize=(12, 5))
plt.plot(df[DATE_COLUMN], df[TARGET])
plt.xlabel("Date")
plt.ylabel(TARGET)
plt.title("Target Over Time")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3. TIME FEATURES
# ============================================================

df["year"] = df[DATE_COLUMN].dt.year
df["month"] = df[DATE_COLUMN].dt.month
df["day"] = df[DATE_COLUMN].dt.day
df["day_of_week"] = df[DATE_COLUMN].dt.dayofweek

df["lag_1"] = df[TARGET].shift(1)
df["lag_7"] = df[TARGET].shift(7)
df["lag_14"] = df[TARGET].shift(14)

# shift first so the current target is not included
df["rolling_mean_7"] = (
    df[TARGET]
    .shift(1)
    .rolling(7)
    .mean()
)

df["rolling_mean_14"] = (
    df[TARGET]
    .shift(1)
    .rolling(14)
    .mean()
)

df = df.dropna().reset_index(drop=True)

print(df.head())

In [ ]:
# ============================================================
# 4. CHRONOLOGICAL SPLIT
# ============================================================

train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

FEATURES = [
    "year",
    "month",
    "day",
    "day_of_week",
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_mean_14",
]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print(len(train_df), len(val_df), len(test_df))

In [ ]:
# ============================================================
# 5. NAIVE BASELINE
# ============================================================

naive_predictions = val_df["lag_1"]

print(
    "Naive MAE:",
    mean_absolute_error(y_val, naive_predictions)
)

print(
    "Naive RMSE:",
    np.sqrt(
        mean_squared_error(
            y_val,
            naive_predictions
        )
    )
)

In [ ]:
# ============================================================
# 6. MODELS
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

In [ ]:
# ============================================================
# 7. EVALUATION
# ============================================================

def evaluate_model(model, X, y, model_name):
    predictions = model.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    mape = mean_absolute_percentage_error(y, predictions)

    print(f"\n========== {model_name} ==========")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")

    return {
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
    }

results = []

for name, model in models.items():
    results.append(
        evaluate_model(model, X_val, y_val, name)
    )

results_df = pd.DataFrame(results).sort_values("rmse")
print(results_df)

In [ ]:
# ============================================================
# 8. FORECAST PLOT
# ============================================================

BEST_MODEL_NAME = results_df.iloc[0]["model"]
best_model = models[BEST_MODEL_NAME]

val_predictions = best_model.predict(X_val)

plt.figure(figsize=(12, 5))
plt.plot(
    val_df[DATE_COLUMN],
    y_val.values,
    label="Actual"
)
plt.plot(
    val_df[DATE_COLUMN],
    val_predictions,
    label="Predicted"
)
plt.title(f"Validation Forecast - {BEST_MODEL_NAME}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9. FINAL TEST
# ============================================================

test_predictions = best_model.predict(X_test)

print("Selected model:", BEST_MODEL_NAME)
print("Test MAE :", mean_absolute_error(y_test, test_predictions))
print(
    "Test RMSE:",
    np.sqrt(mean_squared_error(y_test, test_predictions))
)
print(
    "Test MAPE:",
    mean_absolute_percentage_error(y_test, test_predictions)
)